# Bronze Layer — API Ingestion via Databricks (All 17 Entities)

**Day 7 | Equivalent of ADF `pl_bronze_api_master_v4` + `pl_bronze_api_ingest_v4`**

Replicates the full ADF v4 metadata-driven pipeline logic in pure Python:
- Reads entity config from ADLS (`bronze/config/pipeline_metadata_config.json`)
- Authenticates to VoltGrid API using credentials from Key Vault
- Per-entity: reads watermark, paginates the API, writes JSON pages to Bronze ADLS
- Writes audit row per entity (always), advances watermark only on success
- All 17 entities run via `ThreadPoolExecutor` (parallel — same as ADF ForEach)

---

### ADF → Databricks activity mapping

| ADF Activity | Databricks equivalent |
|---|---|
| `act_get_username / act_get_password` | `dbutils.secrets.get(scope, key)` |
| `act_api_login` | `requests.post(auth_url, json=payload)` |
| `act_set_token` | Python variable `token` |
| `act_set_ingestion_date` | `datetime.now(UTC).strftime('%Y-%m-%d')` |
| `act_get_watermark` | Read per-entity CSV from ADLS via service principal |
| `act_set_watermark` | `epoch` if full, else value from CSV |
| `act_get_total_pages` | GET page 1, read `pagination.total_pages` |
| `act_paginate` (Until loop) | `for page in range(1, total_pages+1)` |
| `act_copy_entity_page` | GET page N, upload JSON bytes to ADLS |
| `act_write_audit` | Append CSV row to `bronze/audit/pipeline_audit.csv` |
| `act_write_watermark` | Overwrite `bronze/audit/watermark_<entity>.csv` |
| Master ForEach (parallel) | `ThreadPoolExecutor(max_workers=17)` |

---

### Bronze output structure (identical to ADF)

```
bronze/
├── config/pipeline_metadata_config.json
├── audit/
│     ├── pipeline_audit.csv           ← append-only history (all entities, all runs)
│     ├── watermark_payments.csv       ← per-entity watermark, overwritten on success
│     └── watermark_<entity>.csv × 17
└── api/
      └── <entity>/ingestion_date=<date>/page_<N>.json
```

---

### `load_type` parameter

| Value | Watermark used | Effect |
|---|---|---|
| `full` | `1900-01-01T00:00:00Z` (epoch) | API returns all records — full historical load |
| `incremental` | Value from `watermark_<entity>.csv` | API returns only records updated after last run |

## Cell 1 — Install dependencies and imports

`requests` is pre-installed on Databricks Runtime. `azure-storage-file-datalake` is also available. No `%pip install` needed on DBR 12+.

In [ ]:
import json
import io
import csv
import uuid
import requests
import concurrent.futures
from datetime import datetime, timezone
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

print("Imports OK")

## Cell 2 — Read `load_type` widget parameter

Passed by the Databricks Job as a parameter. Default is `incremental`.

| `load_type` | Behaviour |
|---|---|
| `full` | Uses epoch watermark — fetches all records for all entities |
| `incremental` | Uses per-entity watermark CSV — fetches only new/updated records |

In [ ]:
dbutils.widgets.text("load_type", "incremental", "Load Type (full / incremental)")
LOAD_TYPE = dbutils.widgets.get("load_type").strip().lower()

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"Invalid load_type='{LOAD_TYPE}'. Must be 'full' or 'incremental'.")

print(f"load_type : {LOAD_TYPE}")

## Cell 3 — Configuration constants

All constants mirror what is hardcoded in the ADF pipeline or its linked services.

In [ ]:
# Key Vault secret scope (Databricks secret scope backed by kv-ev-intelligence-dev)
KV_SCOPE         = "kv-ev-scope"

# API
API_BASE_URL     = "https://ev-project-navy-mu.vercel.app"
AUTH_ENDPOINT    = f"{API_BASE_URL}/api/auth/login/"
EPOCH_WATERMARK  = "1900-01-01T00:00:00Z"   # full load — API returns all records

# ADLS Bronze
STORAGE_ACCOUNT  = "evdatalakedev"
CONTAINER        = "bronze"
ADLS_URL         = f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net"

# ADLS paths
CONFIG_PATH      = "config/pipeline_metadata_config.json"
AUDIT_CSV_PATH   = "audit/pipeline_audit.csv"
WATERMARK_DIR    = "audit"
API_DATA_DIR     = "api"

# Audit: pipeline name written to pipeline_audit.csv (matches ADF value)
PIPELINE_NAME    = "pl_bronze_api_ingest_databricks_v1"

print(f"API base     : {API_BASE_URL}")
print(f"ADLS account : {STORAGE_ACCOUNT}")
print(f"Container    : {CONTAINER}")
print(f"load_type    : {LOAD_TYPE}")

## Cell 4 — Authenticate to ADLS via Service Principal

Reads Service Principal credentials from Key Vault via the Databricks secret scope.
Returns a `DataLakeServiceClient` used for all ADLS read/write operations.

ADF equivalent: `ls_adls_bronze` linked service using Managed Identity. Here we use
the project Service Principal stored in Key Vault (same SP used by the full_load script).

In [ ]:
TENANT_ID     = dbutils.secrets.get(scope=KV_SCOPE, key="adls-tenant-id")
CLIENT_ID     = dbutils.secrets.get(scope=KV_SCOPE, key="adls-client-id")
CLIENT_SECRET = dbutils.secrets.get(scope=KV_SCOPE, key="adls-client-secret")

credential = ClientSecretCredential(
    tenant_id     = TENANT_ID,
    client_id     = CLIENT_ID,
    client_secret = CLIENT_SECRET
)

adls_client  = DataLakeServiceClient(account_url=ADLS_URL, credential=credential)
fs_client    = adls_client.get_file_system_client(CONTAINER)

print(f"ADLS authenticated — OK (account: {STORAGE_ACCOUNT}, container: {CONTAINER})")

## Cell 5 — Helper functions

Small reusable functions for ADLS read/write and watermark handling.
All ADLS operations go through `fs_client` created in Cell 4.

In [ ]:
def adls_read_text(path):
    """Read a text file from ADLS Bronze container."""
    fc = fs_client.get_file_client(path)
    return fc.download_file().readall().decode("utf-8")


def adls_write_bytes(path, data: bytes, overwrite=True):
    """Write bytes to ADLS Bronze container, creating parent directories automatically."""
    fc = fs_client.get_file_client(path)
    fc.upload_data(data, overwrite=overwrite, length=len(data))


def adls_append_csv_row(path, row: dict, fieldnames: list):
    """
    Append one CSV row to an existing file in ADLS.
    Reads current content, appends the row, overwrites the file.
    Thread-safe per-entity because each entity has its own watermark file;
    pipeline_audit.csv is written after all threads complete.
    """
    try:
        existing = adls_read_text(path)
    except Exception:
        # File does not exist yet — create with header
        existing = ",".join(fieldnames) + "\n"

    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=fieldnames, quoting=csv.QUOTE_ALL, lineterminator="\n")
    writer.writerow(row)
    new_content = existing.rstrip("\n") + "\n" + buf.getvalue()
    adls_write_bytes(path, new_content.encode("utf-8"))


def read_watermark(entity_name):
    """Read per-entity watermark value from bronze/audit/watermark_<entity>.csv."""
    path = f"{WATERMARK_DIR}/watermark_{entity_name}.csv"
    try:
        content = adls_read_text(path)
        reader  = csv.DictReader(io.StringIO(content))
        row     = next(reader)
        return row["watermark_value"]
    except Exception as e:
        print(f"  [{entity_name}] WARNING: Could not read watermark ({e}) — using epoch")
        return EPOCH_WATERMARK


def write_watermark(entity_name, timestamp_utc):
    """Overwrite per-entity watermark CSV with current UTC timestamp."""
    path    = f"{WATERMARK_DIR}/watermark_{entity_name}.csv"
    content = (
        "watermark_value,entity_name,updated_at\n"
        f"{timestamp_utc},{entity_name},{timestamp_utc}\n"
    )
    adls_write_bytes(path, content.encode("utf-8"))


print("Helper functions defined — OK")

## Cell 6 — Read entity metadata config from ADLS

ADF equivalent: `act_read_metadata` Lookup activity reading `bronze/config/pipeline_metadata_config.json`.

Returns list of 17 entity dicts with `entity_name`, `api_path`, `page_size`, `enabled`.

In [ ]:
config_json = adls_read_text(CONFIG_PATH)
all_entities = json.loads(config_json)

# Filter to enabled entities only
entities = [e for e in all_entities if e.get("enabled", True)]

print(f"Entities loaded: {len(entities)}")
for e in entities:
    print(f"  {e['entity_name']:<20}  path={e['api_path']:<30}  page_size={e['page_size']}")

## Cell 7 — VoltGrid API authentication

ADF equivalent: `act_get_username` + `act_get_password` (Key Vault WebActivity) + `act_api_login` (POST WebActivity).

One login call is shared by all entities — the same token is reused across all 17 parallel ingestion threads.

In [ ]:
# ADF: act_get_username + act_get_password — read from Key Vault MSI
# Databricks: read from Databricks secret scope backed by Key Vault
USERNAME = dbutils.secrets.get(scope=KV_SCOPE, key="voltgrid-username")
PASSWORD = dbutils.secrets.get(scope=KV_SCOPE, key="voltgrid-password")

# ADF: act_api_login — POST to /api/auth/login/
resp = requests.post(
    AUTH_ENDPOINT,
    json    = {"username": USERNAME, "password": PASSWORD},
    headers = {"Content-Type": "application/json"},
    timeout = 30
)
resp.raise_for_status()

# ADF: act_set_token — store in variable v_token
TOKEN = resp.json()["token"]
AUTH_HEADER = {"Authorization": f"Token {TOKEN}"}

print(f"VoltGrid API authenticated — OK")
print(f"Token : [REDACTED — {len(TOKEN)} chars]")

## Cell 8 — Per-entity ingestion function

This function replicates the full ADF child pipeline `pl_bronze_api_ingest_v4` for one entity:

```
read watermark → set watermark (full/incremental) → get total_pages
→ loop pages → write JSON to ADLS → write audit row → write watermark (on success)
```

Returns a result dict used by Cell 9 for the final summary.

In [ ]:
AUDIT_FIELDNAMES = [
    "pipeline_name", "entity_name", "load_type", "watermark_value",
    "ingestion_date", "total_pages", "status", "pipeline_run_id", "run_timestamp"
]


def ingest_entity(entity: dict, ingestion_date: str, run_id: str) -> dict:
    """
    Full ingestion pipeline for one entity — mirrors pl_bronze_api_ingest_v4.

    Returns dict: {entity_name, status, total_pages, pages_written, error}
    """
    entity_name = entity["entity_name"]
    api_path    = entity["api_path"]
    page_size   = entity["page_size"]

    status      = "failed"
    total_pages = 0
    pages_written = 0
    error_msg   = None

    try:
        # ── act_get_watermark ─────────────────────────────────────────────────
        csv_watermark = read_watermark(entity_name)

        # ── act_set_watermark (full vs incremental) ───────────────────────────
        # ADF: IF p_load_type == 'full' THEN '1900-01-01T00:00:00Z' ELSE csv value
        watermark = EPOCH_WATERMARK if LOAD_TYPE == "full" else csv_watermark

        print(f"  [{entity_name}] load={LOAD_TYPE}  watermark={watermark}")

        # ── act_get_total_pages ───────────────────────────────────────────────
        # GET page 1 to read pagination.total_pages
        page1_url = (
            f"{API_BASE_URL}{api_path}"
            f"?page=1&page_size={page_size}&updated_after={watermark}"
        )
        r1 = requests.get(page1_url, headers=AUTH_HEADER, timeout=60)
        r1.raise_for_status()
        page1_data  = r1.json()
        total_pages = page1_data["pagination"]["total_pages"]

        print(f"  [{entity_name}] total_pages={total_pages}")

        # ── act_paginate (Until loop) → act_copy_entity_page ─────────────────
        for page in range(1, total_pages + 1):
            if page == 1:
                # Reuse already-fetched page 1 data — no extra API call
                page_data = page1_data
            else:
                url  = (
                    f"{API_BASE_URL}{api_path}"
                    f"?page={page}&page_size={page_size}&updated_after={watermark}"
                )
                resp = requests.get(url, headers=AUTH_HEADER, timeout=60)
                resp.raise_for_status()
                page_data = resp.json()

            # Write JSON to ADLS: bronze/api/<entity>/ingestion_date=<date>/page_<N>.json
            adls_path = f"{API_DATA_DIR}/{entity_name}/ingestion_date={ingestion_date}/page_{page}.json"
            adls_write_bytes(adls_path, json.dumps(page_data).encode("utf-8"))
            pages_written += 1

        # ── act_set_status_success ────────────────────────────────────────────
        status = "succeeded"
        print(f"  [{entity_name}] DONE — {pages_written} page(s) written")

    except Exception as e:
        # ── act_set_status_failed ─────────────────────────────────────────────
        status    = "failed"
        error_msg = str(e)
        print(f"  [{entity_name}] FAILED — {error_msg}")

    finally:
        # ── act_write_audit — always runs (Succeeded + Failed dependsOn) ──────
        run_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        audit_row = {
            "pipeline_name":   PIPELINE_NAME,
            "entity_name":     entity_name,
            "load_type":       LOAD_TYPE,
            "watermark_value": watermark if status != "failed" or pages_written > 0 else EPOCH_WATERMARK,
            "ingestion_date":  ingestion_date,
            "total_pages":     str(total_pages),
            "status":          status,
            "pipeline_run_id": run_id,
            "run_timestamp":   run_timestamp
        }
        # Audit rows are collected in a thread-safe list and written by Cell 9
        # after all threads complete — avoids concurrent ADLS append conflicts.

    return {
        "entity_name":   entity_name,
        "status":        status,
        "total_pages":   total_pages,
        "pages_written": pages_written,
        "watermark":     watermark if "watermark" in dir() else EPOCH_WATERMARK,
        "error":         error_msg,
        "audit_row":     audit_row,
        "run_timestamp": audit_row["run_timestamp"]
    }


print("ingest_entity() function defined — OK")

## Cell 9 — Run all entities in parallel

ADF equivalent: `act_foreach_entity` ForEach with `isSequential: false`, `batchCount: 20`.

`ThreadPoolExecutor(max_workers=17)` runs all 17 entity ingestions concurrently.
Each thread is independent — a failure in one entity does not stop others.

After all threads complete, audit rows are written to ADLS sequentially (safe, no concurrent writes).
Watermark is advanced only for entities that succeeded.

In [ ]:
# ADF: act_set_ingestion_date — formatDateTime(utcNow(), 'yyyy-MM-dd')
now            = datetime.now(timezone.utc)
ingestion_date = now.strftime("%Y-%m-%d")
run_id         = str(uuid.uuid4())   # ADF: pipeline().RunId

print(f"Ingestion date : {ingestion_date}")
print(f"Run ID         : {run_id}")
print(f"Entities       : {len(entities)}")
print(f"load_type      : {LOAD_TYPE}")
print("\nStarting parallel ingestion...\n")

results = []

# ADF: ForEach (parallel, batchCount=20)
with concurrent.futures.ThreadPoolExecutor(max_workers=17) as executor:
    futures = {
        executor.submit(ingest_entity, entity, ingestion_date, run_id): entity["entity_name"]
        for entity in entities
    }
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        results.append(result)

print("\nAll entities complete. Writing audit rows and watermarks...")

# ── act_write_audit — always, for every entity ────────────────────────────────
# Collected after all threads complete to avoid concurrent ADLS write conflicts
for r in results:
    try:
        adls_append_csv_row(AUDIT_CSV_PATH, r["audit_row"], AUDIT_FIELDNAMES)
        print(f"  [audit] {r['entity_name']} — written")
    except Exception as e:
        print(f"  [audit] {r['entity_name']} — FAILED to write audit row: {e}")

# ── act_write_watermark — only on success ─────────────────────────────────────
# ADF: act_write_watermark dependsOn act_set_status_success (Succeeded)
wm_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
for r in results:
    if r["status"] == "succeeded":
        try:
            write_watermark(r["entity_name"], wm_timestamp)
            print(f"  [watermark] {r['entity_name']} → {wm_timestamp}")
        except Exception as e:
            print(f"  [watermark] {r['entity_name']} — FAILED to write watermark: {e}")

print("\nAudit and watermark writes complete.")

## Cell 10 — Run summary

Prints a per-entity table identical in spirit to the ADF Monitor panel.
Raises an exception if any entity failed — causes the Databricks Job run to be marked Failed and triggers email alert.

In [ ]:
succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 70)
print("BRONZE API INGESTION — RUN SUMMARY")
print("=" * 70)
print(f"load_type      : {LOAD_TYPE}")
print(f"ingestion_date : {ingestion_date}")
print(f"run_id         : {run_id}")
print(f"entities total : {len(results)}")
print(f"succeeded      : {len(succeeded)}")
print(f"failed         : {len(failed)}")
print()
print(f"  {'Entity':<22} {'Status':<12} {'Pages':<8} {'Watermark used'}")
print(f"  {'-'*22} {'-'*12} {'-'*8} {'-'*30}")
for r in sorted(results, key=lambda x: x["entity_name"]):
    marker = "OK" if r["status"] == "succeeded" else "FAIL"
    print(f"  [{marker}] {r['entity_name']:<20} {r['status']:<12} {r['pages_written']:<8} {r['watermark']}")
    if r["error"]:
        print(f"        Error: {r['error']}")
print()
print(f"Audit rows written to  : bronze/{AUDIT_CSV_PATH}")
print(f"Watermarks updated     : {len(succeeded)} entity file(s)")
print("=" * 70)

# Raise if any entity failed — Job marks run Failed and sends email alert
if failed:
    raise Exception(
        f"{len(failed)} entity(ies) failed: "
        + ", ".join(r['entity_name'] for r in failed)
        + " — check output above for details."
    )